In [ ]:
!pip install transformers datasets sentence-transformers faiss-cpu gradio evaluate

import json
import numpy as np
import torch
import gradio as gr
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModel, pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from tqdm import tqdm

# Load & preprocess SQuAD dataset
def load_and_preprocess_squad_v2():
    dataset = load_dataset('squad_v2')
    contexts = []
    questions = []
    answers = []

    for example in dataset['train']:
        if example['answers']['text']:  # Filter out unanswerable questions
            context = example['context']
            question = example['question']
            answer = example['answers']['text'][0]
            contexts.append(context)
            questions.append(question)
            answers.append(answer)

    return contexts, questions, answers

# Load and preprocess the data
contexts, questions, answers = load_and_preprocess_squad_v2()

# Encode contexts using Sentence-BERT (SBERT)
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all contexts
print("Encoding contexts...")
context_embeddings = sbert_model.encode(contexts, convert_to_tensor=True)

# Define a function to find the most relevant context for a given question
def find_most_relevant_context(question, context_embeddings, contexts):
    question_embedding = sbert_model.encode(question, convert_to_tensor=True)
    # Calculate cosine similarities
    similarities = cosine_similarity(question_embedding.cpu().numpy().reshape(1, -1), context_embeddings.cpu().numpy())
    # Get the index of the most similar context
    best_match_index = np.argmax(similarities)
    return contexts[best_match_index]

# Load a BERT model for question-answering
qa_model_name = 'bert-large-uncased-whole-word-masking-finetuned-squad'
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

# Define a function to extract the answer using the QA model
def extract_answer(question, context):
    # Limit the maximum length to 512 tokens (BERT's limit)
    inputs = qa_tokenizer.encode_plus(
        question,
        context,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    input_ids = inputs["input_ids"].tolist()[0]

    # Get the model's answer scores
    outputs = qa_model(**inputs)
    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    # Get the most likely beginning and end of the answer
    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1

    # Convert the tokens back to string
    answer = qa_tokenizer.convert_tokens_to_string(
        qa_tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end])
    )
    return answer

# Step 5: Create an interactive chat interface using Gradio
def chat_interface(question):
    context = find_most_relevant_context(question, context_embeddings, contexts)
    answer = extract_answer(question, context)
    return f"Question: {question}\n\nContext: {context}\n\nAnswer: {answer}"

# Create Gradio interface
iface = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs="text",
    title="SQuAD v2.0 Chatbot",
    description="Ask any question and get answers based on the SQuAD v2.0 dataset!"
)

# Launch the interface
iface.launch()

# Step 6: Evaluation code for testing the chatbot's performance
def evaluate_on_squad_v2():
    dev_set = load_dataset('squad_v2', split='validation')
    f1_scores = []
    exact_matches = []

    for example in tqdm(dev_set):
        if example['answers']['text']:  # Only evaluate on answerable questions
            question = example['question']
            true_answer = example['answers']['text'][0]
            context = find_most_relevant_context(question, context_embeddings, contexts)
            predicted_answer = extract_answer(question, context)

            # Calculate exact match
            exact_match = int(predicted_answer.strip().lower() == true_answer.strip().lower())
            exact_matches.append(exact_match)

            # Calculate F1 score
            true_tokens = set(true_answer.lower().split())
            predicted_tokens = set(predicted_answer.lower().split())
            common_tokens = true_tokens.intersection(predicted_tokens)
            if len(common_tokens) == 0:
                f1 = 0.0
            else:
                precision = len(common_tokens) / len(predicted_tokens)
                recall = len(common_tokens) / len(true_tokens)
                f1 = 2 * (precision * recall) / (precision + recall)
            f1_scores.append(f1)

    avg_f1 = np.mean(f1_scores)
    avg_exact_match = np.mean(exact_matches)

    print(f"Average F1 Score: {avg_f1:.2f}")
    print(f"Average Exact Match: {avg_exact_match:.2f}")

evaluate_on_squad_v2()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 429.9 kB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 4

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7cee51374c0a000e5d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


 77%|███████▋  | 9111/11873 [3:28:49<1:41:19,  2.20s/it]